## P31
- Ok this will kinda be the first big test if ALOHA will work for use
- For let's say 10 episodes, I want to render -> 3 camera view (jpgs are fine), and control signals (GT and pred I think)
- Will need to noodle a bit on how I want to visualize control signals
- 7x2 grid of time series probably wouldn't be too bad
- Could obviously put multiple ones on a single axis in different colors - that probably wouldn't be too bad either

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image
import copy
import math
import time

import torch
import torch.nn.functional as F

# openpi imports
from openpi.training import config as _config
from openpi.policies import policy_config as _policy_config
from openpi.policies import aloha_policy
from openpi.shared import download
from openpi import transforms as _transforms
from openpi.shared import normalize as _normalize
from openpi.models import model as _model


In [ ]:
joint_labels = [
    "L joint 0 (waist)", "L joint 1 (shoulder)", "L joint 2 (elbow)",
    "L joint 3 (forearm)", "L joint 4 (wrist angle)", "L joint 5 (wrist rotate)",
    "L gripper",
    "R joint 0 (waist)", "R joint 1 (shoulder)", "R joint 2 (elbow)",
    "R joint 3 (forearm)", "R joint 4 (wrist angle)", "R joint 5 (wrist rotate)",
    "R gripper",
]

In [ ]:
def frame_to_obs(frame):
    """Convert a HuggingFace dataset frame to the observation dict
    that the policy expects (post-RepackTransform format)."""
    def to_chw(img):
        arr = np.array(img)  # PIL -> HWC (H, W, 3)
        return arr.transpose(2, 0, 1)  # -> CHW (3, H, W)
    
    return {
        "images": {
            "cam_high": to_chw(frame["observation.images.cam_high"]),
            "cam_left_wrist": to_chw(frame["observation.images.cam_left_wrist"]),
            "cam_right_wrist": to_chw(frame["observation.images.cam_right_wrist"]),
        },
        "state": np.array(frame["observation.state"]),
        "prompt": "uncap the pen",
    }

## Load Dataset

In [ ]:
ds = load_dataset("physical-intelligence/aloha_pen_uncap_diverse", split="train")
print(f"Total rows: {len(ds)}")
print(f"Columns: {ds.column_names}")
print(f"Episode indices: {sorted(set(ds['episode_index']))}")

In [ ]:
# Pick an episode and grab all its frames
EPISODE_IDX = 0

ep_indices = np.array(ds["episode_index"])
mask = ep_indices == EPISODE_IDX
row_indices = np.where(mask)[0]
episode = ds.select(row_indices)

# Pick a frame in the middle of the episode
FRAME_IDX = 150
frame = episode[FRAME_IDX]

In [ ]:
# Display camera views
cam_names = ["cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, cam_names):
    img = frame[f"observation.images.{name}"]
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.suptitle(f"Episode {EPISODE_IDX}, Frame {FRAME_IDX} (t={frame['timestamp']:.2f}s)", fontsize=14)
plt.tight_layout()
plt.show()

## Render images

In [ ]:
import os
from tqdm import tqdm
import matplotlib.cm as cm

CHILL_BROWN='#948979'

In [ ]:
save_dir='data/robots/p31'

In [ ]:
EPISODE_IDX = 0

save_dir_full=save_dir+'/'+str(EPISODE_IDX).zfill(3)
os.makedirs(save_dir_full, exist_ok=True)
os.makedirs(save_dir_full+'/left_wrist', exist_ok=True)
os.makedirs(save_dir_full+'/right_wrist', exist_ok=True)
os.makedirs(save_dir_full+'/low', exist_ok=True)
os.makedirs(save_dir_full+'/high', exist_ok=True)
os.makedirs(save_dir_full+'/gt', exist_ok=True)
os.makedirs(save_dir_full+'/pred', exist_ok=True)
os.makedirs(save_dir_full+'/pred_combined', exist_ok=True)
os.makedirs(save_dir_full+'/pred_tall', exist_ok=True)
os.makedirs(save_dir_full+'/transparent', exist_ok=True)

ep_indices = np.array(ds["episode_index"])
mask = ep_indices == EPISODE_IDX
row_indices = np.where(mask)[0]
episode = ds.select(row_indices)


In [ ]:
for FRAME_IDX in tqdm(range(episode.num_rows)):
    frame = episode[FRAME_IDX]
    im=frame['observation.images.cam_left_wrist']
    im.save(save_dir_full+'/left_wrist/'+str(FRAME_IDX).zfill(3)+'.jpg')

    im=frame['observation.images.cam_right_wrist']
    im.save(save_dir_full+'/right_wrist/'+str(FRAME_IDX).zfill(3)+'.jpg')

    im=frame['observation.images.cam_low']
    im.save(save_dir_full+'/low/'+str(FRAME_IDX).zfill(3)+'.jpg')

    im=frame['observation.images.cam_high']
    im.save(save_dir_full+'/high/'+str(FRAME_IDX).zfill(3)+'.jpg')

In [ ]:
im.size

## Load model

In [ ]:
config = _config.get_config("pi0_aloha_pen_uncap")

# Point to converted PyTorch checkpoint
checkpoint_dir = "data/robots/openpi/pi0_aloha_pen_uncap_pytorch"

print(f"Model type: {config.model.model_type}")
print(f"Action dim: {config.model.action_dim}")
print(f"Action horizon: {config.model.action_horizon}")

# Create the trained policy — auto-detects PyTorch format
policy = _policy_config.create_trained_policy(config, checkpoint_dir)
print(f"Policy created successfully!")
print(f"PyTorch model: {policy._is_pytorch_model}")
print(f"Device: {policy._pytorch_device}")
print(f"Number of input transforms: {len(policy._input_transform.transforms)}")
print(f"Number of output transforms: {len(policy._output_transform.transforms)}")

# Quick reference inference to make sure the model works
obs_test = frame_to_obs(frame)
result_test = policy.infer(obs_test)
print(f"Actions shape: {result_test['actions'].shape}")
print(f"Inference time: {result_test['policy_timing']['infer_ms']:.1f} ms")

- Hmm so for the time series, I feel like there's kinda two broad options
- Windowed for full duration
- My gut is let's try full duration
- Let me decouple my problems a bit here and first just use the GT motions to work on a coupel vis options
- THen I'll do the same ish for the preds

In [ ]:
CHILL_BROWN = '#948979'
labels = ["Waist", "Shoulder", "Elbow", "Forearm", "Wrist angle", "Wrist rotate", "Gripper" ]
labels_all = ["L Waist", "L Shoulder", "L Elbow", "L Forearm", "L Wrist angle", "L Wrist rotate", "L Gripper",
              "R Waist", "R Shoulder", "R Elbow", "R Forearm", "R Wrist angle", "R Wrist rotate", "R Gripper"]

actions = np.array([episode[i]["action"] for i in range(episode.num_rows)]) #Slower than I would expect!

In [ ]:
for FRAME_IDX in tqdm(range(episode.num_rows)):
    plt.clf()
    fig = plt.figure(0, (12, 4), facecolor='k')
    ax = fig.add_subplot(111, facecolor='k')
    
    lines = plt.plot(actions[:FRAME_IDX, :7], linewidth=3)
    for line, label in zip(lines, labels):
        line.set_label(label)
    
    ax.tick_params(colors=CHILL_BROWN, labelsize=16)
    for spine in ax.spines.values():
        spine.set_edgecolor(CHILL_BROWN)
    ax.xaxis.label.set_color(CHILL_BROWN)
    ax.yaxis.label.set_color(CHILL_BROWN)
    plt.ylim(-1.3, 1.3)
    
    leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11,
                    facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)
    
    plt.tight_layout()

    plt.savefig(save_dir_full+'/gt/'+str(FRAME_IDX).zfill(3)+'.png', dpi=200)

- Ok, now how do we want to do predictions? 

In [ ]:
all_actions=[]
for FRAME_IDX in tqdm(range(episode.num_rows)):

    rng = np.random.RandomState(42)
    fixed_noise = rng.randn(50, 32).astype(np.float32)
    
    frame = episode[FRAME_IDX]
    obs_ref = frame_to_obs(frame)
    result_ref = policy.infer(obs_ref, noise=fixed_noise)
    all_actions.append(result_ref['actions'])

Ok let's try a simple "run and replace" every 5 steps. 

In [ ]:
pred_actions_merged=[] 

step_size=5
for i in range(0, episode.num_rows, step_size):
    pred_actions_merged.append(all_actions[i][:step_size])

pred_actions_merged=np.concatenate(pred_actions_merged)

In [ ]:
for FRAME_IDX in tqdm(range(episode.num_rows)):
    plt.clf()
    fig = plt.figure(0, (12, 8), facecolor='k')
    ax = fig.add_subplot(2,1,1, facecolor='k')
    
    lines = plt.plot(actions[:FRAME_IDX, :7], linewidth=3)
    for line, label in zip(lines, labels):
        line.set_label(label)
    
    ax.tick_params(colors=CHILL_BROWN, labelsize=16)
    for spine in ax.spines.values():
        spine.set_edgecolor(CHILL_BROWN)
    ax.xaxis.label.set_color(CHILL_BROWN)
    ax.yaxis.label.set_color(CHILL_BROWN)
    plt.ylim(-1.3, 1.3)
    
    leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11,
                    facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)

    ax = fig.add_subplot(2,1,2, facecolor='k')
    lines = ax.plot(actions[:FRAME_IDX, 7:], linewidth=3)
    ax.tick_params(colors=CHILL_BROWN, labelsize=16)
    for spine in ax.spines.values():
        spine.set_edgecolor(CHILL_BROWN)
    ax.xaxis.label.set_color(CHILL_BROWN)
    ax.yaxis.label.set_color(CHILL_BROWN)
    plt.ylim(-1.3, 1.3)
    
    plt.tight_layout()

    plt.savefig(save_dir_full+'/pred/'+str(FRAME_IDX).zfill(3)+'.png', dpi=200)

In [ ]:
FRAME_IDX=200
plt.clf()
fig = plt.figure(0, (16, 4), facecolor='k')
ax = fig.add_subplot(1,1,1, facecolor='k')

lines = plt.plot(actions[:FRAME_IDX, :], linewidth=3)
for line, label in zip(lines, labels_all):
    line.set_label(label)

ax.tick_params(colors=CHILL_BROWN, labelsize=20)
for spine in ax.spines.values():
    spine.set_edgecolor(CHILL_BROWN)
ax.xaxis.label.set_color(CHILL_BROWN)
ax.yaxis.label.set_color(CHILL_BROWN)
plt.ylim(-1.3, 1.3)

leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=14,
                facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)

# plt.tight_layout()

plt.savefig(save_dir+'/legend_2.png', dpi=600, bbox_inches='tight')

In [ ]:
for FRAME_IDX in tqdm(range(episode.num_rows)):
    plt.clf()
    fig = plt.figure(0, (16, 4), facecolor='k')
    ax = fig.add_subplot(1,1,1, facecolor='k')
    
    lines = plt.plot(actions[:FRAME_IDX, :], linewidth=3)
    for line, label in zip(lines, labels_all):
        line.set_label(label)
    
    ax.tick_params(colors=CHILL_BROWN, labelsize=20)
    for spine in ax.spines.values():
        spine.set_edgecolor(CHILL_BROWN)
    ax.xaxis.label.set_color(CHILL_BROWN)
    ax.yaxis.label.set_color(CHILL_BROWN)
    plt.ylim(-1.3, 1.3)
    
    leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=14,
                    facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)
    
    plt.tight_layout()

    plt.savefig(save_dir_full+'/pred_combined/'+str(FRAME_IDX).zfill(3)+'.png', dpi=200)

In [ ]:
# plt.clf()
# fig = plt.figure(0, (12, 4), facecolor='k')
# ax = fig.add_subplot(1,1,1, facecolor='k')

# lines = plt.plot(actions[:FRAME_IDX, :], linewidth=3)
# for line, label in zip(lines, labels_all):
#     line.set_label(label)

# ax.tick_params(colors=CHILL_BROWN, labelsize=16)
# for spine in ax.spines.values():
#     spine.set_edgecolor(CHILL_BROWN)
# ax.xaxis.label.set_color(CHILL_BROWN)
# ax.yaxis.label.set_color(CHILL_BROWN)
# plt.ylim(-1.3, 1.3)

# leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11,
#                 facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)

# plt.tight_layout()

In [ ]:

plt.clf()
fig = plt.figure(0, (16, 6), facecolor='k')
ax = fig.add_subplot(1,1,1, facecolor='k')

lines = plt.plot(actions[:FRAME_IDX, :], linewidth=3)
for line, label in zip(lines, labels_all):
    line.set_label(label)

ax.tick_params(colors=CHILL_BROWN, labelsize=22)
for spine in ax.spines.values():
    spine.set_edgecolor(CHILL_BROWN)
ax.xaxis.label.set_color(CHILL_BROWN)
ax.yaxis.label.set_color(CHILL_BROWN)
plt.ylim(-1.3, 1.3)

# leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=20,
#                 facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)

# plt.tight_layout()

plt.savefig(save_dir_full+'/pred_tall/'+str(FRAME_IDX).zfill(3)+'.png', dpi=200)

## Transpareent background, small

In [ ]:
FRAME_IDX=250

plt.clf()
fig = plt.figure(0, (16, 6), facecolor='k')
ax = fig.add_subplot(1,1,1, facecolor='k')

lines = plt.plot(actions[:FRAME_IDX, (2,5,4,6, 8)], linewidth=5)
for line, label in zip(lines, labels_all):
    line.set_label(label)

ax.tick_params(colors=CHILL_BROWN, labelsize=22)
for spine in ax.spines.values():
    spine.set_edgecolor(CHILL_BROWN)
ax.xaxis.label.set_color(CHILL_BROWN)
ax.yaxis.label.set_color(CHILL_BROWN)
plt.ylim(-1.0, 1.3)
plt.axis('off')

# leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=20,
#                 facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)

# plt.tight_layout()

plt.savefig(save_dir_full+'/transparent/'+str(FRAME_IDX).zfill(3)+'.png', dpi=200, transparent=True)



In [ ]:
for FRAME_IDX in tqdm(range(episode.num_rows)):
    plt.clf()
    fig = plt.figure(0, (16, 6), facecolor='k')
    ax = fig.add_subplot(1,1,1, facecolor='k')
    
    lines = plt.plot(actions[:FRAME_IDX, (2,5,4,6, 8)], linewidth=5)
    for line, label in zip(lines, labels_all):
        line.set_label(label)
    
    ax.tick_params(colors=CHILL_BROWN, labelsize=22)
    for spine in ax.spines.values():
        spine.set_edgecolor(CHILL_BROWN)
    ax.xaxis.label.set_color(CHILL_BROWN)
    ax.yaxis.label.set_color(CHILL_BROWN)
    plt.ylim(-1.3, 1.3)
    plt.axis('off')
    
    # leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=20,
    #                 facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)
    
    # plt.tight_layout()
    
    plt.savefig(save_dir_full+'/transparent/'+str(FRAME_IDX).zfill(3)+'.png', dpi=200)

In [ ]:
for FRAME_IDX in tqdm(range(episode.num_rows)):
    plt.clf()
    fig = plt.figure(0, (16, 6), facecolor='k')
    ax = fig.add_subplot(1,1,1, facecolor='k')
    
    lines = plt.plot(actions[:FRAME_IDX, :], linewidth=3)
    for line, label in zip(lines, labels_all):
        line.set_label(label)
    
    ax.tick_params(colors=CHILL_BROWN, labelsize=22)
    for spine in ax.spines.values():
        spine.set_edgecolor(CHILL_BROWN)
    ax.xaxis.label.set_color(CHILL_BROWN)
    ax.yaxis.label.set_color(CHILL_BROWN)
    plt.ylim(-1.0, 1.3)
    
    # leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=20,
    #                 facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)
    
    # plt.tight_layout()
    
    plt.savefig(save_dir_full+'/pred_tall/'+str(FRAME_IDX).zfill(3)+'.png', dpi=200)

In [ ]:
FRAME_IDX

In [ ]:
# plt.clf()
# fig = plt.figure(0, (16, 6), facecolor='k')
# ax = fig.add_subplot(1,1,1, facecolor='k')

# lines = plt.plot(actions[:FRAME_IDX, :], linewidth=3)
# for line, label in zip(lines, labels_all):
#     line.set_label(label)

# ax.tick_params(colors=CHILL_BROWN, labelsize=18)
# for spine in ax.spines.values():
#     spine.set_edgecolor(CHILL_BROWN)
# ax.xaxis.label.set_color(CHILL_BROWN)
# ax.yaxis.label.set_color(CHILL_BROWN)
# plt.ylim(-1.3, 1.3)

# leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=16,
#                 facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)

# plt.tight_layout()

# plt.savefig(save_dir_full+'/pred_tall/'+str(FRAME_IDX).zfill(3)+'.png', dpi=200)

---

In [ ]:
# plt.clf()
# fig = plt.figure(0, (12, 8), facecolor='k')
# ax = fig.add_subplot(2,1,1, facecolor='k')

# lines = plt.plot(actions[:FRAME_IDX, :7], linewidth=3)
# for line, label in zip(lines, labels):
#     line.set_label(label)

# ax.tick_params(colors=CHILL_BROWN, labelsize=16)
# for spine in ax.spines.values():
#     spine.set_edgecolor(CHILL_BROWN)
# ax.xaxis.label.set_color(CHILL_BROWN)
# ax.yaxis.label.set_color(CHILL_BROWN)
# plt.ylim(-1.3, 1.3)

# leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11,
#                 facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)

# ax = fig.add_subplot(2,1,2, facecolor='k')
# lines = ax.plot(actions[:FRAME_IDX, 7:], linewidth=3)
# ax.tick_params(colors=CHILL_BROWN, labelsize=16)
# for spine in ax.spines.values():
#     spine.set_edgecolor(CHILL_BROWN)
# ax.xaxis.label.set_color(CHILL_BROWN)
# ax.yaxis.label.set_color(CHILL_BROWN)
# plt.ylim(-1.3, 1.3)

# plt.tight_layout()


In [ ]:
# fig = plt.figure(0, (12, 4), facecolor='k')
# ax = fig.add_subplot(111, facecolor='k')

# lines = plt.plot(pred_actions_merged[:, :7], linewidth=3)

# # lines = plt.plot(np.arange(10, 60), all_actions[10][:, :7], linewidth=3)

# ax.tick_params(colors=CHILL_BROWN, labelsize=16)
# for spine in ax.spines.values():
#     spine.set_edgecolor(CHILL_BROWN)
# ax.xaxis.label.set_color(CHILL_BROWN)
# ax.yaxis.label.set_color(CHILL_BROWN)
# plt.ylim(-1.3, 1.3)

In [ ]:
# for FRAME_IDX in tqdm(range(episode.num_rows)):
#     plt.clf()

#     fig = plt.figure(0, (12, 4), facecolor='k')
#     ax = fig.add_subplot(111, facecolor='k')
    
#     lines = plt.plot(pred_actions_merged[:, :7], linewidth=3)
#     for line, label in zip(lines, labels):
#         line.set_label(label)
    
#     ax.tick_params(colors=CHILL_BROWN, labelsize=16)
#     for spine in ax.spines.values():
#         spine.set_edgecolor(CHILL_BROWN)
#     ax.xaxis.label.set_color(CHILL_BROWN)
#     ax.yaxis.label.set_color(CHILL_BROWN)
#     plt.ylim(-1.3, 1.3)
    
#     leg = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11,
#                     facecolor='k', edgecolor=CHILL_BROWN, labelcolor=CHILL_BROWN)
    
#     plt.tight_layout()


In [ ]:
# Fix noise for reproducibility (numpy, no JAX dependency)
rng = np.random.RandomState(42)
fixed_noise = rng.randn(50, 32).astype(np.float32)
print(f"fixed_noise: {fixed_noise.shape}, dtype={fixed_noise.dtype}")
print(f"  norm={np.linalg.norm(fixed_noise):.4f}, mean={fixed_noise.mean():.4f}")

# Reference result using policy.infer as a black box
obs_ref = frame_to_obs(frame)
result_ref = policy.infer(obs_ref, noise=fixed_noise)
print(f"\nReference actions: {result_ref['actions'].shape}")
print(f"  range: [{result_ref['actions'].min():.4f}, {result_ref['actions'].max():.4f}]")

In [ ]:
predicted_actions = result_ref["actions"]

In [ ]:
# Gather ground truth actions for the next action_horizon frames
action_horizon = predicted_actions.shape[0]
n_gt = min(action_horizon, len(episode) - FRAME_IDX)

gt_actions = np.array([episode[FRAME_IDX + t]["action"] for t in range(n_gt)])
print(f"Ground truth actions: {gt_actions.shape}")
print(f"Predicted actions: {predicted_actions.shape}")

In [ ]:
# Plot predicted vs ground truth for each joint
fig, axes = plt.subplots(7, 2, figsize=(16, 28), sharex=True)
time_steps = np.arange(n_gt)

for arm_idx, arm_name in enumerate(["Left", "Right"]):
    offset = arm_idx * 7
    for joint_idx in range(7):
        ax = axes[joint_idx, arm_idx]
        dim = offset + joint_idx
        
        ax.plot(time_steps, gt_actions[:n_gt, dim], 'b-', linewidth=2, label='Ground Truth', alpha=0.8)
        ax.plot(time_steps, predicted_actions[:n_gt, dim], 'r--', linewidth=2, label='Predicted', alpha=0.8)
        
        ax.set_ylabel(joint_labels[dim].replace(f"{arm_name[0]} ", ""), fontsize=9)
        ax.grid(True, alpha=0.3)
        if joint_idx == 0:
            ax.set_title(f"{arm_name} Arm", fontsize=12, fontweight='bold')
            ax.legend(fontsize=8)
        if joint_idx == 6:
            ax.set_xlabel("Timestep")

plt.suptitle(f"PyTorch Model: Predicted vs Ground Truth Actions (from frame {FRAME_IDX})", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()